# Chapter 9 — The Twelve Pillars of Production Multi-Agent Systems

**Multi-Agent Analog EDA — PhD-Level Monograph (Notebook Form)**

---

Production-grade **multi-agent systems (MAS)** for analog EDA differ from research demos along twelve orthogonal engineering dimensions. Each pillar is necessary but not sufficient: failures compound across **model integration**, **observability**, **context**, **domain semantics**, **retrieval/compute**, **governance**, **lifecycle**, **operations**, **environments**, **human factors**, **packaging**, and **enterprise glue**.

### Learning objectives

1. **Formalize** BYOM routing as a *policy* over latency–cost–quality constraints with auditable fallbacks.
2. **Instrument** end-to-end traces with **span hierarchies** aligned to EDA tool boundaries (netlist → sim → signoff).
3. **Engineer** typed, budgeted context with **lossy compression** policies grounded in design-critical invariants.
4. **Encode** EDA **ontologies** (devices, DRC/LVS rules, PDK views) as machine-checkable graphs.
5. **Operate** vector/RAG services for **design pattern retrieval** with PDK-aware metadata filters.
6. **Implement** governance primitives: RBAC, immutable audit logs, **IP segmentation**, export-control hooks.
7. **Version** agents and prompts with **CI/CD**, canary/rollback, and **regression suites** on golden designs.
8. **Automate** scaling and **queueing** for heterogeneous simulation farms (SPICE, EM, field solvers).
9. **Isolate** dev/prod via **sandboxes**, **deterministic replay**, and configuration contracts.
10. **Calibrate** human-in-the-loop UX: approvals, overrides, **confidence**, and counterfactual explanations.
11. **Ship** reproducible stacks via **Docker** / Compose with sidecar observability.
12. **Integrate** batch/scheduled runs into legacy EDA flows (Make/LSF/Slurm, design databases).

### Notation

- Design artifact space $\mathcal{A}$ (netlists, constraints, layouts, reports). Agent policy $\pi$ maps traces to actions.
- Model ensemble $\mathcal{M}=\{m_1,\ldots,m_K\}$ with routers $r: \mathcal{X} \to \Delta_K$.
- Trace $T$ is a DAG of **spans** $\{s_i\}$ with attributes (tenant, PDK, tool, job id).

> **Environment note.** All code is **self-contained** (stdlib + numpy/matplotlib; optional networkx if available). **No API keys** or external SaaS calls—LangSmith-style behavior is illustrated with a *local* trace recorder.

---


## Roadmap: the twelve pillars

| # | Pillar | Production intent |
|---|--------|---------------------|
| 1 | Secure model integration (BYOM) | Pluggable LLMs/embeddings with routing & fallbacks |
| 2 | End-to-end observability | Traces, logs, metrics, span trees |
| 3 | Typed context engineering | Structured windows, token budgets, compression |
| 4 | Domain ontology | EDA graphs: devices, rules, PDK views |
| 5 | Vector / compute tool services | Embeddings + metadata filters for design reuse |
| 6 | Governance & security | RBAC, audit, IP zones, export control |
| 7 | Agent lifecycle (CI/CD) | Versioning, A/B, rollback, regression |
| 8 | Operational automation | Autoscale, queues, farm scheduling |
| 9 | Development environments | Sandboxes, deterministic replay, config split |
| 10 | Human-in-the-loop UX | Approvals, overrides, calibration, explanations |
| 11 | Package / release (Docker) | Containers, Compose, reproducibility |
| 12 | Enterprise automation | Legacy flow integration, batch & schedules |

The cells below walk each pillar with **concept → code → figure → EDA callouts**.

---


In [ ]:
# Global setup: dark theme, reproducibility
from __future__ import annotations

import hashlib
import json
import math
import random
import time
import uuid
from collections import defaultdict, deque
from dataclasses import dataclass, field
from datetime import datetime, timezone
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt

RNG = np.random.default_rng(42)

DARK_BG = "#0d1117"
ACCENT = "#58a6ff"
GREEN = "#3fb950"
ORANGE = "#d29922"
RED = "#f85149"
PURPLE = "#bc8cff"
CYAN = "#39d0d0"
MUTED = "#8b949e"

mpl.rcParams.update(
    {
        "figure.facecolor": DARK_BG,
        "axes.facecolor": DARK_BG,
        "axes.edgecolor": "#30363d",
        "axes.labelcolor": "#c9d1d9",
        "text.color": "#c9d1d9",
        "xtick.color": MUTED,
        "ytick.color": MUTED,
        "grid.color": "#21262d",
        "grid.alpha": 0.65,
        "legend.facecolor": "#161b22",
        "legend.edgecolor": "#30363d",
        "font.size": 11,
    }
)

def fig_ax(figsize=(9, 4.5)):
    fig, ax = plt.subplots(figsize=figsize, dpi=120)
    return fig, ax

print("Ready: production MAS didactics (dark matplotlib theme).")


## Pillar 1 — Secure model integration (BYOM)

**Concept.** *Bring-your-own-model* decouples orchestration from any single vendor. A **router** selects among registered backends (local, private endpoint, distilled student) under **SLOs** (max latency, max cost/token). **Fallback chains** preserve liveness: on timeout or policy violation, degrade gracefully while emitting structured errors for audit.

**EDA angle.** Different tasks need different models: *constraint formalization* (long context), *netlist diff summarization* (code-like), *DRC explanation* (short, high precision). **API keys** must be injected via vault sidecars—not notebooks— and scoped per **PDK / foundry** partition.

---


In [ ]:
# Pillar 1 — toy BYOM router + fallback chain + latency/cost plot

@dataclass
class ModelEndpoint:
    name: str
    latency_ms: float
    cost_per_1k: float
    max_ctx: int
    allowed_tenants: frozenset[str]


class ModelRouter:
    # Greedy router: eligible models by tenant & context, then score quality/latency/cost.

    def __init__(self, models: List[ModelEndpoint], weights: Tuple[float, float, float]):
        self.models = models
        self.wq, self.wl, self.wc = weights

    def eligible(self, tenant: str, ctx_tokens: int) -> List[ModelEndpoint]:
        return [m for m in self.models if tenant in m.allowed_tenants and m.max_ctx >= ctx_tokens]

    def score(self, m: ModelEndpoint) -> float:
        # toy: quality proxy inverse latency
        q = 1.0 / (1e-3 + m.latency_ms / 1000.0)
        return self.wq * q - self.wl * (m.latency_ms / 1000.0) - self.wc * m.cost_per_1k

    def choose(self, tenant: str, ctx_tokens: int) -> Optional[ModelEndpoint]:
        cand = self.eligible(tenant, ctx_tokens)
        if not cand:
            return None
        return max(cand, key=self.score)

    def fallback_chain(self, tenant: str, ctx_tokens: int) -> List[str]:
        cand = sorted(self.eligible(tenant, ctx_tokens), key=self.score, reverse=True)
        return [m.name for m in cand]


models = [
    ModelEndpoint("gpt-class-remote", 900, 3.0, 128_000, frozenset({"acme", "internal"})),
    ModelEndpoint("private-70b", 350, 0.8, 32_000, frozenset({"acme", "foundry_x"})),
    ModelEndpoint("distill-8b-local", 40, 0.05, 8_000, frozenset({"acme", "internal", "sandbox"})),
]

router = ModelRouter(models, weights=(1.0, 0.15, 0.25))
print("Fallback chain (acme, 6k ctx):", router.fallback_chain("acme", 6000))

ctx_grid = np.linspace(500, 40_000, 200)
chosen = []
for c in ctx_grid:
    m = router.choose("acme", int(c))
    chosen.append(m.name if m else "NONE")

fig, ax = fig_ax((10, 4))
for i, name in enumerate({x for x in chosen}):
    mask = np.array([x == name for x in chosen])
    ax.fill_between(ctx_grid, 0, mask.astype(float), step="mid", alpha=0.35, label=name)
ax.set_xlabel("Context length (tokens)")
ax.set_yticks([])
ax.set_title("Toy BYOM routing: model choice vs context budget (tenant=acme)")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, axis="x")
plt.tight_layout()
plt.show()


## Pillar 2 — End-to-end observability

**Concept.** Observability unifies **logs** (discrete events), **metrics** (time series aggregates), and **traces** (causal DAGs). For agents, traces should mirror **tool boundaries**: each SPICE job, extractor invocation, and LLM completion is a **span** with parent/child links—enabling latency attribution and failure blast-radius analysis.

**LangSmith-style mapping (conceptual).** A *run* ≈ span; *thread* ≈ user/session; *project* ≈ product line. Export traces to your SIEM with **PII redaction** and **foundry-safe** attribute whitelists.

**EDA angle.** Attribute spans with `pdk_id`, `corner_set_hash`, `cell_name`, `license_feature`, and `simulator_build`. This makes regressions **queryable**: "all failures where Spectre exit code ≠ 0 under monte batch 7."

---


In [ ]:
# Pillar 2 — in-process span tree + Gantt-style visualization

@dataclass
class Span:
    id: str
    name: str
    parent: Optional[str]
    start: float
    end: float
    attrs: Dict[str, Any] = field(default_factory=dict)


class Tracer:
    def __init__(self):
        self.spans: List[Span] = []

    def span(self, name: str, parent: Optional[str], fn: Callable[[], Any], **attrs) -> Any:
        sid = str(uuid.uuid4())[:8]
        t0 = time.perf_counter()
        try:
            return fn()
        finally:
            t1 = time.perf_counter()
            self.spans.append(Span(sid, name, parent, t0, t1, attrs))


tr = Tracer()
root = "root"


def leaf_sleep(ms: float):
    time.sleep(ms / 1000.0)


def run_iter():
    # In production, parent ids link llm → extract/drc; here we tag depth explicitly for the Gantt.
    tr.span("llm.plan", root, lambda: leaf_sleep(12), model="private-70b")
    tr.span("spice.batch", root, lambda: leaf_sleep(55), corners=64, solver="spectre")
    tr.span("extract.starrc", root, lambda: leaf_sleep(28), tech="n5", under="llm")
    tr.span("drc.calibre", root, lambda: leaf_sleep(18), deck="metal_fill", under="llm")


tr.span("agent.iteration", root, run_iter, cell="LDO_CORE", pdk="N5FF")

# Normalize times for plot
t0 = min(s.start for s in tr.spans)
rows = []
for s in sorted(tr.spans, key=lambda x: x.start):
    depth = {"agent.iteration": 0, "llm.plan": 1, "spice.batch": 1, "extract.starrc": 2, "drc.calibre": 2}.get(
        s.name, 1
    )
    rows.append((depth, s))

fig, ax = fig_ax((10, 4.5))
y_labels = []
for i, (d, s) in enumerate(sorted(rows, key=lambda r: r[1].start)):
    y = i
    y_labels.append(s.name)
    x0 = (s.start - t0) * 1000
    x1 = (s.end - t0) * 1000
    color = ACCENT if "llm" in s.name else GREEN if "spice" in s.name else ORANGE
    ax.barh(y, x1 - x0, left=x0, height=0.65, color=color, alpha=0.85, edgecolor="#30363d")

ax.set_yticks(range(len(y_labels)))
ax.set_yticklabels(y_labels, fontsize=9)
ax.set_xlabel("ms from trace start")
ax.set_title("Toy span timeline (LangSmith-like hierarchical runs)")
ax.grid(True, axis="x")
plt.tight_layout()
plt.show()

print("Sample span attrs:", {s.name: s.attrs for s in tr.spans})


## Pillar 3 — Typed context engineering

**Concept.** Treat the LLM context as a **typed product** $\mathcal{C} = \bigoplus_\ell \mathcal{C}_\ell$: system policy, retrieved docs, tool outputs, and *working memory*. Enforce **schemas** (JSON or protobuf) and a **token budget** allocator $B = \sum_\ell B_\ell$. **Compression** is a monotone map $\psi_\ell: \mathcal{C}_\ell \to \mathcal{C}_\ell'$ that preserves invariant predicates $I$ (e.g., supply rails, bias currents).

**EDA angle.** Never drop **signoff thresholds** or **absolute max Vds** accidentally. Prefer *structured summarization* of waveforms (overshoot %, settling time) over raw CSV dumps.

---


In [ ]:
# Pillar 3 — typed context pack + budgeted truncation + compression ratio plot

@dataclass
class ContextChunk:
    role: str
    kind: str
    text: str
    priority: int  # higher = keep longer


def approx_tokens(s: str) -> int:
    return max(1, len(s) // 4)


def pack_context(chunks: List[ContextChunk], budget: int) -> Tuple[List[ContextChunk], Dict[str, int]]:
    ordered = sorted(chunks, key=lambda c: c.priority, reverse=True)
    out: List[ContextChunk] = []
    used = 0
    stats = {"raw_tokens": 0, "packed_tokens": 0}
    for c in ordered:
        t = approx_tokens(c.text)
        stats["raw_tokens"] += t
        if used + t <= budget:
            out.append(c)
            used += t
        else:
            room = budget - used
            if room > 24:
                clipped = c.text[: room * 4] + " …[truncated]"
                out.append(ContextChunk(c.role, c.kind + "|trunc", clipped, c.priority))
                used = budget
            break
    stats["packed_tokens"] = used
    return out, stats


chunks = [
    ContextChunk("system", "policy", "You must preserve VDD=0.9±0.05 and IM3<-40dBc summaries.", 100),
    ContextChunk("user", "spec", "Target: 2mA bias, 10MHz GBW, 60deg PM.", 90),
    ContextChunk("tool", "spice_log", "TRAN: 20000 lines of raw output ... " + ("noise " * 4000), 40),
    ContextChunk("retrieval", "app_note", "Stability with cap at gate: Z(s) ... " + ("detail " * 1200), 55),
]

packed, st = pack_context(chunks, budget=350)
print("Packed kinds:", [p.kind for p in packed])
print("Stats:", st)

budgets = np.linspace(120, 800, 60)
ratios = []
for b in budgets:
    _, s = pack_context(chunks, int(b))
    ratios.append(s["packed_tokens"] / max(1, s["raw_tokens"]))

fig, ax = fig_ax((9, 4))
ax.plot(budgets, ratios, color=PURPLE, lw=2)
ax.set_xlabel("Token budget")
ax.set_ylabel("Packed / raw token mass")
ax.set_title("Context budgeting: fraction retained vs budget (toy)")
ax.grid(True)
plt.tight_layout()
plt.show()


## Pillar 4 — Domain ontology

**Concept.** An **ontology** is a typed graph $(\mathcal{O}, \sqsubseteq, \mathcal{R})$ of concepts and relations. For EDA, instantiate **device taxonomy** (mos/rf/fet variants), **constraint kinds** (electrical, geometric, reliability), and **rule ontologies** linking DRC decks to *physical meaning*.

**EDA angle.** Ontologies enable *consistent tool routing*: if a query mentions **"ESD clamp"**, map to a subgraph of devices, allowed topologies, and required checks (IV curve, thermal, CDM).

---


In [ ]:
# Pillar 4 — tiny ontology graph visualization (networkx optional, else matplotlib fallback)

nodes = {
    "Device": (0.0, 0.0),
    "MOSFET": (-1.2, -1.0),
    "PassFET": (-2.0, -2.0),
    "ESD": (1.5, -1.0),
    "Constraint": (0.0, 1.5),
    "Electrical": (-1.0, 2.6),
    "Geometric": (1.2, 2.6),
    "DRC_Rule": (2.8, 0.2),
}

edges = [
    ("Device", "MOSFET"),
    ("MOSFET", "PassFET"),
    ("Device", "ESD"),
    ("Device", "Constraint"),
    ("Constraint", "Electrical"),
    ("Constraint", "Geometric"),
    ("Geometric", "DRC_Rule"),
]

fig, ax = fig_ax((8, 6))
for n, (x, y) in nodes.items():
    ax.scatter([x], [y], s=900, color=ACCENT, alpha=0.25, edgecolors=ACCENT, linewidths=2)
    ax.text(x, y, n, ha="center", va="center", fontsize=9, color="#e6edf3")

for a, b in edges:
    x1, y1 = nodes[a]
    x2, y2 = nodes[b]
    ax.annotate(
        "",
        xy=(x2, y2),
        xytext=(x1, y1),
        arrowprops=dict(arrowstyle="->", color=MUTED, lw=1.4, connectionstyle="arc3,rad=0.08"),
    )

ax.set_title("EDA ontology fragment: devices → constraints → rules")
ax.axis("off")
plt.tight_layout()
plt.show()


## Pillar 5 — Vector / compute tool services

**Concept.** **Embeddings** map artifacts $a \in \mathcal{A}$ to vectors for *similarity search*. Production systems wrap a **vector database** with **metadata filters** (PDK, process node, block type). **Hybrid retrieval** combines BM25 on identifiers with vectors on *topologies*.

**EDA angle.** Index **netlist motifs**, **testbench templates**, and **post-layout failure reports**. Filter by `foundry_allowed=True` to avoid leaking restricted PDK text.

---


In [ ]:
# Pillar 5 — toy embedding search (hashing trick) + precision@k simulation

def embed_text(s: str, dim: int = 32) -> np.ndarray:
    v = np.zeros(dim, dtype=np.float64)
    for tok in s.lower().split():
        h = int(hashlib.sha256(tok.encode()).hexdigest(), 16)
        v[h % dim] += 1.0
    n = np.linalg.norm(v) + 1e-9
    return v / n


corpus = [
    ("tb_ldo_line_transient", "LDO line transient testbench with ESL package model"),
    ("tb_opamp_ac_noise", "AC sweep and noise summary for two-stage opamp"),
    ("tb_ring_osc_pvt", "PVT corners for ring oscillator period measurement"),
    ("tb_adc_inl_dnl", "histogram test for SAR ADC INL/DNL"),
    ("tb_esd_clamp_iv", "TLP-style IV snapshot for ESD clamp device"),
]

queries = [
    "I need a transient test for regulator with package inductance",
    "corner simulation oscillator frequency",
]

dim = 48
emb_c = np.stack([embed_text(t, dim) for _, t in corpus])
fig, axes = plt.subplots(1, len(queries), figsize=(11, 4), sharey=True)
if len(queries) == 1:
    axes = [axes]

for ax, q in zip(axes, queries):
    eq = embed_text(q, dim)
    sims = emb_c @ eq
    idx = np.argsort(-sims)
    ax.barh([corpus[i][0] for i in idx[::-1]], sims[idx[::-1]], color=CYAN, alpha=0.85)
    ax.set_title(q[:42] + "…")
    ax.set_xlabel("cosine similarity (toy)")
    ax.grid(True, axis="x")

plt.suptitle("Vector retrieval over synthetic EDA corpus (hash embeddings)", y=1.02)
plt.tight_layout()
plt.show()


## Pillar 6 — Governance & security

**Concept.** **Governance** constrains *who can cause what in which zone*. Implement **RBAC** (roles → permissions), **ABAC** (attributes like export jurisdiction), and **immutable audit trails** (append-only event log with hash chaining). **IP protection** uses segmentation, encryption at rest, and **watermarked** exports.

**EDA angle.** **Foundry NDAs** and **EAR/ITAR** may prohibit cloud regions or foreign model hosts—encode as policy predicates evaluated *before* routing (Pillar 1).

---


In [ ]:
# Pillar 6 — RBAC + append-only audit chain

@dataclass
class AuditEvent:
    ts: str
    actor: str
    action: str
    resource: str
    prev_hash: str

    def fingerprint(self) -> str:
        payload = json.dumps(
            [self.ts, self.actor, self.action, self.resource, self.prev_hash], sort_keys=True
        )
        return hashlib.sha256(payload.encode()).hexdigest()


class AuditLog:
    def __init__(self):
        self.chain: List[AuditEvent] = []

    def append(self, actor: str, action: str, resource: str) -> None:
        prev = self.chain[-1].fingerprint() if self.chain else "GENESIS"
        ev = AuditEvent(datetime.now(timezone.utc).isoformat(), actor, action, resource, prev)
        self.chain.append(ev)


ROLES = {
    "designer": {"read:cell", "run:sim"},
    "lead": {"read:cell", "run:sim", "approve:layout"},
    "admin": {"read:cell", "run:sim", "approve:layout", "export:golden"},
    "contractor": {"read:cell"},
}


def allowed(role: str, perm: str) -> bool:
    return perm in ROLES.get(role, set())


log = AuditLog()
for actor, role, perm, res in [
    ("alice@acme", "designer", "run:sim", "LDO_CORE:tran"),
    ("bob@acme", "lead", "approve:layout", "LDO_CORE:layout@v3"),
    ("eve@acme", "contractor", "export:golden", "LDO_CORE"),
]:
    ok = allowed(role, perm)
    log.append(actor, f"{perm}:{'OK' if ok else 'DENIED'}", res)

hashes = [e.fingerprint()[:10] for e in log.chain]
fig, ax = fig_ax((9, 3))
ax.step(range(len(hashes)), [int(h, 16) % 10_000 for h in hashes], where="mid", color=ORANGE)
ax.set_xticks(range(len(log.chain)))
ax.set_xticklabels([e.action for e in log.chain], rotation=25, ha="right")
ax.set_title("Toy audit chain: evolving fingerprint (lower 16 bits mod 10k)")
ax.grid(True)
plt.tight_layout()
plt.show()

print("Last event:", log.chain[-1])


## Pillar 7 — Agent lifecycle (CI/CD)

**Concept.** Agents are **software artifacts**: prompts, tool manifests, policies, and model pins. **Version** them like libraries (semver + git SHA). **A/B tests** route traffic fractions to candidate policies; **rollback** restores last-known-good. **Regression tests** run *golden* designs (small netlists with expected metric bands) on every commit.

**EDA angle.** Gate releases on **PPA slack deltas** across a **golden bench suite** (LDO, BG, opamp, ADC front-end). Fail CI if DRC count regresses or if simulation **non-convergence** rate increases.

---


In [ ]:
# Pillar 7 — traffic split + rollback state machine (toy)

@dataclass
class AgentRelease:
    version: str
    pass_rate: float  # golden suite


def choose_arm(arms: List[AgentRelease], traffic: Dict[str, float], rng: np.random.Generator) -> str:
    r = rng.random()
    acc = 0.0
    for a in arms:
        acc += traffic.get(a.version, 0.0)
        if r <= acc:
            return a.version
    return arms[-1].version


arms = [
    AgentRelease("v1.2.0", 0.97),
    AgentRelease("v1.3.0-canary", 0.93),
]
traffic = {"v1.2.0": 0.9, "v1.3.0-canary": 0.1}

n = 5_000
picked = [choose_arm(arms, traffic, RNG) for _ in range(n)]
canary_frac = np.mean([p == "v1.3.0-canary" for p in picked])

# Rollback rule: if canary pass_rate < threshold, shift traffic to 100% stable
threshold = 0.94
stable = arms[0]
canary = arms[1]
post_traffic = (
    {"v1.2.0": 1.0, "v1.3.0-canary": 0.0}
    if canary.pass_rate < threshold
    else traffic
)

fig, ax = fig_ax((9, 4))
labels = list(post_traffic.keys())
vals = [post_traffic[k] for k in labels]
ax.bar(labels, vals, color=[GREEN if "canary" not in k else ORANGE for k in labels], edgecolor="#30363d")
ax.set_ylim(0, 1.05)
ax.set_title(f"Traffic after rollback check (canary pass_rate={canary.pass_rate}, threshold={threshold})")
ax.set_ylabel("Traffic fraction")
ax.grid(True, axis="y")
plt.tight_layout()
plt.show()

print(f"Empirical canary traffic ~ {canary_frac:.3f} (expected 0.100)")
print("Post-policy:", post_traffic)


## Pillar 8 — Operational automation

**Concept.** Production MAS sits on **compute queues** (batch schedulers). **Autoscaling** adjusts worker pools from queue depth $Q$ and age $A$: scale $\propto \max(0, Q - Q_0) + \lambda A$. **Resource classes** separate *interactive* sizing loops from *batch* corner farms.

**EDA angle.** Tag jobs with **license tokens** (Spectre, Calibre) and **memory footprints** so the scheduler avoids thrashing. Prefer **backpressure** when PDK file servers saturate.

---


In [ ]:
# Pillar 8 — queue + autoscaler simulation (discrete time)

T = 120
arrival_rate = 0.42
service_rate_base = 0.38
Q = 0.0
Qs = []
workers = 3.0
worker_hist = []

for t in range(T):
    Q = max(0.0, Q + RNG.poisson(arrival_rate) - workers * RNG.poisson(service_rate_base))
    Qs.append(Q)
    target = 3 + 0.35 * max(0.0, Q - 5) + 0.02 * Q
    workers = float(np.clip(0.85 * workers + 0.15 * target, 2, 12))
    worker_hist.append(workers)

fig, ax = fig_ax((10, 4))
ax.plot(Qs, color=ACCENT, label="Queue depth (toy)")
ax.set_xlabel("time step")
ax2 = ax.twinx()
ax2.plot(worker_hist, color=GREEN, ls="--", label="workers (smoothed)")
ax.set_ylabel("depth")
ax2.set_ylabel("workers")
ax.set_title("Autoscaler smoothing vs queue depth (illustrative)")
ax.grid(True)
fig.legend(loc="upper right", bbox_to_anchor=(0.88, 0.92))
plt.tight_layout()
plt.show()


## Pillar 9 — Development environments

**Concept.** **Sandboxes** isolate writes (no production netlist paths). **Deterministic replay** fixes RNG seeds, pins tool versions, and records I/O hashes so the same trace reproduces. Split **configs**: `dev` enables verbose traces and mock simulators; `prod` enforces quotas and redacts secrets.

**EDA angle.** Replay requires **PDK revision pins** and **Monte Carlo seed files**. Treat simulator **option blocks** as part of the reproducibility surface.

---


In [ ]:
# Pillar 9 — config split + deterministic replay fingerprint

@dataclass
class RunConfig:
    env: str
    mock_sim: bool
    trace_verbose: bool
    pdk_rev: str


def effective_config(env: str) -> RunConfig:
    base = RunConfig(env=env, mock_sim=False, trace_verbose=False, pdk_rev="N5FF_r1.3")
    if env == "dev":
        return RunConfig(env="dev", mock_sim=True, trace_verbose=True, pdk_rev="N5FF_r1.3")
    return base


def run_fingerprint(cfg: RunConfig, design: str, seed: int) -> str:
    payload = json.dumps([cfg.__dict__, design, seed], sort_keys=True)
    return hashlib.sha256(payload.encode()).hexdigest()[:16]


cfg_prod = effective_config("prod")
cfg_dev = effective_config("dev")
fp1 = run_fingerprint(cfg_prod, "LDO_CORE", 123)
fp2 = run_fingerprint(cfg_dev, "LDO_CORE", 123)

fig, ax = fig_ax((8, 2))
ax.axis("off")
rows = [["env", cfg_prod.env, cfg_dev.env], ["mock_sim", cfg_prod.mock_sim, cfg_dev.mock_sim], ["trace", cfg_prod.trace_verbose, cfg_dev.trace_verbose], ["fp", fp1, fp2]]
for i, row in enumerate(rows):
    for j, cell in enumerate(row):
        ax.text(j * 0.33 + 0.02, 0.75 - i * 0.22, str(cell), fontsize=11, color="#e6edf3", family="monospace")
ax.set_title("dev vs prod configuration + replay fingerprint (toy table)", loc="left")
plt.tight_layout()
plt.show()


## Pillar 10 — Human-in-the-loop UX

**Concept.** **Approval gates** pause subgraphs until a qualified human accepts risk. **Overrides** let experts correct tool outputs with audited rationale. **Confidence calibration** maps model scores to empirical error rates (reliability diagrams). **Explanations** should cite *artifacts* (netlist line spans, violated DRC checks), not generic text.

**EDA angle.** Gate **layout commits** and **constraint relaxations**. Show **corner coverage** and **worst-slack** histograms beside the model's recommendation.

---


In [ ]:
# Pillar 10 — reliability diagram (calibration) + approval threshold

bins = np.linspace(0, 1, 11)
conf = RNG.uniform(0, 1, 4000)
# Toy: under-confident model → accuracy > confidence on average
acc = (RNG.random(4000) < 0.2 + 0.7 * conf).astype(float)

bin_idx = np.clip((conf * 10).astype(int), 0, 9)
mean_conf = [conf[bin_idx == k].mean() if np.any(bin_idx == k) else np.nan for k in range(10)]
acc_rate = [acc[bin_idx == k].mean() if np.any(bin_idx == k) else np.nan for k in range(10)]

fig, ax = fig_ax((5, 5))
ax.plot([0, 1], [0, 1], color=MUTED, ls="--", label="perfect calibration")
ax.plot(mean_conf, acc_rate, color=PURPLE, marker="o", label="empirical accuracy")
ax.set_xlabel("Mean predicted confidence (bin)")
ax.set_ylabel("Empirical accuracy")
ax.set_title("Reliability diagram (synthetic)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

THRESH = 0.72
needs_approval = lambda p: p < THRESH
print("approval?", {0.9: needs_approval(0.9), 0.65: needs_approval(0.65)})


## Pillar 11 — Package / release (Docker)

**Concept.** **Containerize** the orchestrator, tool adapters, and sidecars (vault agent, trace exporter). **Docker Compose** wires dependencies on a single host; Kubernetes extends this to clusters. Pin **base images** and use multi-stage builds to minimize attack surface.

**EDA angle.** Mount **PDK** and **license servers** as read-only volumes or delegated sockets; never bake secrets into layers. Use **same UID/GID** maps as farm batch users for file ownership.

The next cell stores a **reference** `compose.yaml` fragment as data (not executed)—illustrative only.

---


In [ ]:
# Pillar 11 — illustrative Compose fragment (string) + service topology sketch

COMPOSE_SNIPPET = '''
# Reference only — adapt to your org
services:
  orchestrator:
    image: analog-mas:1.4.0
    read_only: true
    environment:
      - ENV=prod
    volumes:
      - type: bind
        source: /secure/pdk
        target: /pdk:ro
  trace-sidecar:
    image: otel-collector:0.97.0
    volumes:
      - ./otel.yaml:/etc/otel.yaml:ro
  queue-bridge:
    image: slurm-submit-helper:0.2.1
'''

print(COMPOSE_SNIPPET)

fig, ax = fig_ax((9, 3.5))
ax.set_xlim(0, 10)
ax.set_ylim(0, 3)
ax.axis("off")
boxes = [
    (1, 1.2, "orchestrator\n(agent graph)"),
    (4.2, 1.2, "tool adapters\n(SPICE/DRC)"),
    (7.2, 1.2, "sidecars\n(traces/vault)"),
]
for x, y, txt in boxes:
    ax.add_patch(plt.Rectangle((x - 0.55, y - 0.45), 1.5, 1.0, fill=False, edgecolor=ACCENT, lw=2))
    ax.text(x + 0.2, y, txt, ha="center", va="center", fontsize=9, color="#e6edf3")
ax.annotate("", xy=(3.5, 1.7), xytext=(2.5, 1.7), arrowprops=dict(arrowstyle="->", color=MUTED))
ax.annotate("", xy=(6.5, 1.7), xytext=(5.7, 1.7), arrowprops=dict(arrowstyle="->", color=MUTED))
ax.set_title("Docker Compose mental model: orchestrator ↔ adapters ↔ sidecars")
plt.tight_layout()
plt.show()


## Pillar 12 — Enterprise automation

**Concept.** Integrate agents with **existing** EDA drivers: Makefile/CMake flows, **LSF/Slurm/Grid Engine**, design databases (CLI or REST), and **artifact stores** (S3-compatible). **Batch** runs sweep blocks; **schedules** trigger nightly regressions and PDK migration checks.

**EDA angle.** Emit standard **log formats** consumed by legacy dashboards. Preserve **run IDs** that match farm job arrays for cross-linking in observability (Pillar 2).

---


In [ ]:
# Pillar 12 — batch scheduler stub + Gantt of scheduled design runs

@dataclass
class BatchJob:
    name: str
    start_h: int
    dur_h: int
    pool: str


jobs = [
    BatchJob("LDO_corners", 0, 8, "spice_farm"),
    BatchJob("ADC_dnl", 2, 6, "spice_farm"),
    BatchJob("DRC_full_chip", 8, 4, "signoff"),
    BatchJob("ESD_batch", 12, 5, "special"),
    BatchJob("PDK_smoke", 20, 3, "spice_farm"),
]

fig, ax = fig_ax((10, 4))
pool_color = {"spice_farm": ACCENT, "signoff": GREEN, "special": ORANGE}
for i, j in enumerate(jobs):
    ax.barh(i, j.dur_h, left=j.start_h, height=0.55, color=pool_color[j.pool], alpha=0.85, edgecolor="#30363d")
    ax.text(j.start_h + 0.15, i, j.name, va="center", fontsize=9, color="#0d1117")

ax.set_yticks(range(len(jobs)))
ax.set_yticklabels([j.pool for j in jobs])
ax.set_xlabel("hour of day (toy)")
ax.set_title("Enterprise batch schedule (pools as row labels)")
ax.grid(True, axis="x")
plt.tight_layout()
plt.show()


## Synthesis — closing the demo→production gap

The twelve pillars are **coupled**: weak governance (6) undermines BYOM (1); poor observability (2) blocks reliable CI/CD (7); absent ontology (4) poisons retrieval (5) and HITL trust (10). A practical rollout orders **(2,6,1)** first—*see everything, control access, then plug models*—then **(3,4,5)** for semantic correctness, and finally **(7–12)** for scale and enterprise fit.

**Checklist for analog EDA teams**

- Pin **PDK + simulator + extractor** versions in every trace.
- Treat **prompts/policies** as versioned artifacts with golden **block regressions**.
- Run **vector indices** per IP zone with export-control metadata.
- Containerize with **read-only** PDK mounts and externalized secrets.

---

### Further reading (topics)

OpenTelemetry tracing, hash-chained audit logs, reliability diagrams & temperature scaling, Slurm job arrays, reproducible builds (Nix), policy-as-code (OPA), and differential privacy for shared logs.

---
